# 4GB GPU에서 QLoRA 학습하기

이 노트북은 완성된 스크립트를 호출하지 않고 QLoRA의 각 단계를 직접 실행합니다.
처음에는 `RUN_MODE = "smoke"`로 1 step만 확인하고, 성공하면 커널을 재시작한 뒤 `"full"`로 바꾸세요.

## 1. 프로젝트 경로와 실험 선택

- `TASK`: `reply` 또는 `fridge`
- `RUN_MODE`: `smoke` 또는 `full`
- 다른 task를 학습하기 전에는 커널을 재시작해 VRAM을 비웁니다.

In [1]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in candidates if (path / "day05").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TASK = "reply"       # "reply" 또는 "fridge"
RUN_MODE = "smoke"  # 먼저 "smoke", 이후 "full"
print("project:", PROJECT_ROOT)
print("task:", TASK, "/ mode:", RUN_MODE)

project: /home/student/llm-practice/c5-slm
task: reply / mode: smoke


## 2. GPU와 라이브러리 확인

4GB 환경에서는 학습 전 여유 VRAM이 3.2GiB 이상인 것이 안전합니다. 부족하면 다른 Jupyter 커널을 종료하세요.

In [2]:
import torch, transformers, peft, trl, bitsandbytes

assert torch.cuda.is_available(), "CUDA GPU를 찾지 못했습니다."
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM free/total: {free_bytes/1024**3:.2f}/{total_bytes/1024**3:.2f} GiB")
print("bf16 supported:", torch.cuda.is_bf16_supported())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__, "/ trl:", trl.__version__, "/ bitsandbytes:", bitsandbytes.__version__)

/home/student/llm-practice/c5-slm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM free/total: 3.22/4.00 GiB
bf16 supported: True
torch: 2.13.0+cu130
transformers: 5.16.1
peft: 0.20.0 / trl: 1.12.0 / bitsandbytes: 0.50.2


## 3. 원본 데이터 확인

학습 40개와 검증 8개를 읽습니다. 테스트 5개는 학습에 넣지 않고 마지막 비교에만 사용합니다.

In [3]:
from datasets import load_dataset
from day05.life_assistant.config import MODEL_ID, TASKS, training_pair, user_text

task = TASKS[TASK]
train_raw = load_dataset("json", data_files=str(task["train"]), split="train")
val_raw = load_dataset("json", data_files=str(task["val"]), split="train")
print("train:", len(train_raw), "/ val:", len(val_raw))
train_raw[0]

train: 40 / val: 8


{'relation': '직장 동료',
 'situation': '부탁받은 자료 정리를 이번 주 금요일까지 끝내기 어렵다',
 'intent': '다음 주 화요일까지 연기 요청',
 'answer': '부탁하신 자료 정리는 이번 주 금요일까지 마무리하기 어려울 것 같습니다. 가능하시다면 다음 주 화요일까지 전달드려도 괜찮을까요?'}

## 4. Mi:dm 기본 template 문제 확인

Mi:dm의 기본 chat template는 약 500토큰의 system prompt를 자동으로 붙입니다. 원본 강의의 `max_length=512`에서는 assistant 정답이 잘릴 수 있으므로 짧은 prompt/completion 형식을 사용합니다.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

first = train_raw[0]
messages = [
    {"role": "user", "content": user_text(TASK, first)},
    {"role": "assistant", "content": first["answer"]},
]
default_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True)["input_ids"]
print("기본 chat template token 수:", len(default_ids))

pair = training_pair(TASK, first)
compact_ids = tokenizer(pair["prompt"] + pair["completion"], add_special_tokens=False).input_ids
print("compact prompt/completion token 수:", len(compact_ids))
print("\n[PROMPT]\n", pair["prompt"])
print("\n[COMPLETION]\n", pair["completion"])

기본 chat template token 수: 561
compact prompt/completion token 수: 67

[PROMPT]
 <|begin_of_text|><|start_header_id|>user<|end_header_id|>

관계: 직장 동료
상황: 부탁받은 자료 정리를 이번 주 금요일까지 끝내기 어렵다
의도: 다음 주 화요일까지 연기 요청<|eot_id|><|start_header_id|>assistant<|end_header_id|>



[COMPLETION]
 부탁하신 자료 정리는 이번 주 금요일까지 마무리하기 어려울 것 같습니다. 가능하시다면 다음 주 화요일까지 전달드려도 괜찮을까요?<|eot_id|>


In [5]:
train = train_raw.map(lambda row: training_pair(TASK, row), remove_columns=train_raw.column_names)
val = val_raw.map(lambda row: training_pair(TASK, row), remove_columns=val_raw.column_names)

lengths = [
    len(tokenizer(row["prompt"] + row["completion"], add_special_tokens=False).input_ids)
    for row in train
]
print(f"token min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}")
assert max(lengths) <= 192

Map: 100%|██████████| 8/8 [00:00<00:00, 2216.86 examples/s]

token min=50, max=70, mean=61.3


## 5. 4-bit base 모델 로드

NF4는 정규분포 형태의 가중치를 4-bit로 표현하고, double quantization은 양자화 상수까지 다시 압축합니다. base 가중치는 학습하지 않습니다.

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

torch.cuda.empty_cache()
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    dtype=torch.bfloat16,
    local_files_only=True,
)
base.config.use_cache = False
print(f"base 로드 후 allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

Loading weights: 100%|██████████| 434/434 [00:04<00:00, 100.18it/s]

base 로드 후 allocated: 1.43 GiB


## 6. LoRA와 학습 설정

4GB에 맞춰 rank 8을 사용하고 attention의 `q_proj`, `v_proj`만 학습합니다. micro batch는 1이며 4번 누적해 유효 batch 4를 만듭니다.

In [7]:
from peft import LoraConfig
from trl import SFTConfig

lora = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

is_smoke = RUN_MODE == "smoke"
output_dir = task["adapter"].with_name(task["adapter"].name + ("-notebook-smoke" if is_smoke else ""))
config = SFTConfig(
    output_dir=str(output_dir),
    seed=42,
    num_train_epochs=3,
    max_steps=1 if is_smoke else -1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_length=192,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_cache=False,
    optim="paged_adamw_8bit",
    completion_only_loss=True,
    logging_steps=1 if is_smoke else 5,
    eval_strategy="no" if is_smoke else "epoch",
    save_strategy="no",
    report_to=[],
)
print("adapter output:", output_dir)

adapter output: /home/student/llm-practice/c5-slm/day05/outputs/adapter-reply-notebook-smoke


## 7. Trainer 생성과 학습 파라미터 확인

전체 23억 파라미터 중 실제로 gradient를 계산하는 LoRA 파라미터 비율을 확인합니다.

In [8]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=base,
    args=config,
    train_dataset=train,
    eval_dataset=None if is_smoke else val,
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

Truncating train dataset: 100%|██████████| 40/40 [00:00<00:00, 7635.38 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 40/40 [00:00<00:00, 13539.84 examples/s]

trainable params: 3,342,336 || all params: 2,308,859,648 || trainable%: 0.1448


## 8. 학습 실행

loss가 내려가는지와 최고 할당 VRAM을 확인합니다. smoke가 성공하면 커널을 재시작하고 `RUN_MODE="full"`로 다시 실행합니다.

In [9]:
import time

torch.cuda.reset_peak_memory_stats()
started = time.time()
result = trainer.train()
elapsed = time.time() - started
peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print(f"elapsed={elapsed:.1f} sec / peak allocated={peak_gib:.2f} GiB")
print(f"train loss={result.training_loss:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
1,2.850094


elapsed=5.5 sec / peak allocated=2.81 GiB
train loss=2.8501


## 9. 로그 확인과 어댑터 저장

In [10]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)
display(history[[column for column in ["epoch", "loss", "eval_loss", "learning_rate"] if column in history]])

trainer.save_model(str(output_dir))
print("saved:", output_dir)

,epoch,loss,learning_rate
0,0.1,2.850094,0.0002
1,0.1,NaN,NaN


saved: /home/student/llm-practice/c5-slm/day05/outputs/adapter-reply-notebook-smoke


## 10. VRAM 정리

다른 task를 학습하거나 비교 노트북을 실행하기 전에는 이 셀을 실행한 뒤 커널을 재시작하는 것이 가장 확실합니다.

In [11]:
import gc

del trainer, base
gc.collect()
torch.cuda.empty_cache()
print(f"cleanup 후 allocated: {torch.cuda.memory_allocated()/1024**3:.3f} GiB")

cleanup 후 allocated: 0.454 GiB
